# NutriVision — Fine-tuning Stage 2 + Optuna (YOLOv11l-seg)

**Konsep Stage 2:**
- Starting weight: `STAGE1_BEST.pt` (head sudah hangat, backbone sudah adapted)
- `freeze=0` → seluruh layer dibuka untuk fine-tuning
- `lr0=0.0001` (10× lebih kecil dari Stage 1) → gentle fine-tuning, hindari catastrophic forgetting
- `warmup_epochs=1.0` → cukup singkat karena weights sudah matang
- Optuna dijalankan **di dalam notebook ini**, setelah Stage 2 baseline selesai
- Optuna search dari `STAGE2_BEST.pt` → cari hyperparams terbaik → final run

**Urutan eksekusi:**
```
Stage 1 best.pt
    ↓
[Section 5] Stage 2 Baseline (freeze=0, lr0=0.0001, 80 epoch)
    ↓ STAGE2_BEST.pt
[Section 6] Optuna Search (N_TRIALS trial, tiap trial 20 epoch)
    ↓ best_params
[Section 7] Final Run (dari STAGE2_BEST.pt + best_params, 80 epoch)
    ↓ FINAL_BEST.pt
[Section 8] Evaluation (val + test)
[Section 9] Artifacts & Download
```

**Input yang dibutuhkan di Kaggle:**
1. Output notebook Stage 1 (berisi `stage1_artifacts.zip` atau `best.pt`)
2. Dataset `clean_dataset` (output data_exploration_repair)

Aktifkan **GPU P100 / T4** sebelum Run All.


## 1. Environment

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
from IPython.display import display, FileLink

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

if importlib.util.find_spec("optuna") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])

try:
    import cv2; cv2.setNumThreads(0)
except Exception:
    pass

from ultralytics import YOLO
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT  = Path("/kaggle/working")

# Slug dataset clean (sama dengan yang dipakai Stage 1)
DATASET_SLUG  = "clean-dataset"

# Slug output notebook Stage 1 — sesuaikan dengan username/notebook kamu
# Format: /kaggle/input/<notebook-output-slug>/
STAGE1_SLUG   = "nutrivision-stage1"   # ← ganti sesuai slug output Stage 1 kamu

# ── Optuna config ─────────────────────────────────────────────────────────────
N_TRIALS         = 15    # jumlah trial Optuna — naikkan jika waktu GPU cukup
OPTUNA_EPOCHS    = 20    # epoch per trial — cukup untuk lihat tren
OPTUNA_PATIENCE  = 8     # early stopping per trial

# ── Stage 2 & Final run config ────────────────────────────────────────────────
STAGE2_EPOCHS  = 80
FINAL_EPOCHS   = 80
IMG_SIZE       = 640
RESUME         = False

# ── Directories ───────────────────────────────────────────────────────────────
RUNS_ROOT = WORK_ROOT / "stage2_runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# ── GPU check ─────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Kaggle: Settings > Accelerator > GPU, lalu Restart & Run All.")

DEVICE        = 0
GPU_NAME      = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
WORKERS       = min(4, max(2, (os.cpu_count() or 4) // 2))

print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {GPU_NAME}  ({GPU_MEMORY_GB:.1f} GB)")
print(f"Workers  : {WORKERS}")
print(f"Optuna   : {optuna.__version__}  |  N_TRIALS={N_TRIALS}  OPTUNA_EPOCHS={OPTUNA_EPOCHS}")


## 2. Dataset discovery

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_files(path, exts=IMAGE_EXTS):
    p = Path(path)
    return sum(1 for f in p.rglob("*") if f.is_file() and f.suffix.lower() in exts) if p.exists() else 0

def find_dataset_root(input_root, slug):
    direct = input_root / slug
    if direct.exists():
        return direct
    for root in sorted(input_root.iterdir()):
        if not root.is_dir():
            continue
        for yaml_path in root.rglob("data.yaml"):
            parent = yaml_path.parent
            if all((parent / s).is_dir() for s in ("train", "valid", "test")):
                return root
    raise FileNotFoundError(
        f"Dataset '{slug}' tidak ditemukan. "
        f"Input: {[str(p) for p in input_root.iterdir() if p.is_dir()]}"
    )

# ── Dataset clean ─────────────────────────────────────────────────────────────
dataset_input = find_dataset_root(INPUT_ROOT, DATASET_SLUG)
all_yamls     = list(dataset_input.rglob("data.yaml"))
if not all_yamls:
    raise FileNotFoundError(f"data.yaml tidak ditemukan di {dataset_input}")

DATA_YAML_SOURCE = sorted(all_yamls, key=lambda p: len(str(p)))[0]
DATASET_ROOT     = DATA_YAML_SOURCE.parent

with DATA_YAML_SOURCE.open("r", encoding="utf-8") as f:
    DATA_CONFIG = yaml.safe_load(f) or {}

CLASS_NAMES = DATA_CONFIG.get("names", [])
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES, key=lambda x: int(x))]
CLASS_NAMES = [str(n) for n in CLASS_NAMES]
NUM_CLASSES = int(DATA_CONFIG.get("nc", len(CLASS_NAMES)))

def resolve(value):
    p = Path(str(value))
    return p if p.is_absolute() else DATASET_ROOT / p

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"  # selalu pakai folder, bukan manifest
VAL_IMAGES   = resolve(DATA_CONFIG.get("val",  "valid/images"))
TEST_IMAGES  = resolve(DATA_CONFIG["test"]) if DATA_CONFIG.get("test") else None

# ── Runtime YAML (absolute paths, tanpa manifest oversampled) ─────────────────
# Stage 2 tidak pakai oversampled manifest — augmentasi copy_paste online sudah cukup
RUNTIME_DATA_YAML = WORK_ROOT / "stage2_runtime_data.yaml"
runtime_config = {
    "path":  str(DATASET_ROOT),
    "train": str(TRAIN_IMAGES),
    "val":   str(VAL_IMAGES),
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
if TEST_IMAGES:
    runtime_config["test"] = str(TEST_IMAGES)
RUNTIME_DATA_YAML.write_text(
    yaml.safe_dump(runtime_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)
DATA_YAML = RUNTIME_DATA_YAML

print(f"Dataset root : {DATASET_ROOT}")
print(f"Train images : {count_files(TRAIN_IMAGES)}")
print(f"Val images   : {count_files(VAL_IMAGES)}")
print(f"Test images  : {count_files(TEST_IMAGES) if TEST_IMAGES else 'N/A'}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"\nRuntime YAML:\n{DATA_YAML.read_text()}")


## 3. Load Stage 1 best.pt

In [ ]:
def find_stage1_best(input_root, stage1_slug):
    """
    Cari best.pt dari output Stage 1. Bisa dalam bentuk:
    1. ZIP stage1_artifacts.zip yang perlu di-extract
    2. File best.pt langsung
    Mencari secara rekursif di semua input.
    """
    # Cari ZIP dulu
    zip_candidates = sorted(input_root.rglob("stage1_artifacts.zip"))
    if zip_candidates:
        extract_dir = WORK_ROOT / "stage1_extracted"
        if not extract_dir.exists():
            extract_dir.mkdir(parents=True)
            with zipfile.ZipFile(zip_candidates[0], "r") as z:
                z.extractall(extract_dir)
            print(f"ZIP extracted: {zip_candidates[0]}")
        best_candidates = sorted(extract_dir.rglob("best.pt"))
        if best_candidates:
            return best_candidates[0]

    # Cari best.pt langsung
    best_candidates = sorted(input_root.rglob("best.pt"))
    if best_candidates:
        # Pilih yang dari stage1 (bukan stage2 jika ada)
        stage1_bests = [p for p in best_candidates if "stage1" in str(p).lower()]
        return stage1_bests[0] if stage1_bests else best_candidates[0]

    raise FileNotFoundError(
        f"Stage 1 best.pt tidak ditemukan di {input_root}.\n"
        f"Pastikan output notebook Stage 1 sudah di-attach sebagai Input."
    )

STAGE1_BEST = find_stage1_best(INPUT_ROOT, STAGE1_SLUG)
print(f"Stage 1 best.pt : {STAGE1_BEST}")
print(f"Size            : {STAGE1_BEST.stat().st_size / 1024**2:.1f} MB")

# Load model
stage1_model = YOLO(str(STAGE1_BEST))
total_params = sum(p.numel() for p in stage1_model.model.parameters())
print(f"\nModel loaded — {total_params:,} params, {len(stage1_model.model.model)} layers")
print(f"Classes: {stage1_model.names}")


## 4. Stage 2 training configuration

### Prinsip utama vs Stage 1

| Parameter | Stage 1 | Stage 2 | Alasan |
|---|---|---|---|
| `freeze` | 11 (backbone frozen) | **0** (semua layer terbuka) | fine-tune seluruh network |
| `lr0` | 0.001 | **0.0001** | 10× lebih kecil — hindari catastrophic forgetting |
| `lrf` | 0.01 | **0.01** | final LR = 1e-6, sangat kecil untuk fine-tuning halus |
| `warmup_epochs` | 3.0 | **1.0** | weights sudah matang, warmup singkat |
| `epochs` | 60 | **80** | lebih lama karena seluruh network perlu adapt |
| `mosaic` | 1.0 | **0.75** | kurangi sedikit — model sudah kenal variasi, fokus ke refinement |
| `copy_paste` | 0.30 | **0.40** | naikkan — bantu kelas minoritas di full fine-tuning |
| `mixup` | 0.05 | **0.15** | naikkan — regularisasi lebih kuat saat full fine-tuning |
| `patience` | 20 | **25** | lebih sabar — plateau lebih sering di full fine-tuning |


In [ ]:
STAGE2_CONFIG = {
    # ── Dataset & hardware ────────────────────────────────────────────────────
    "data":    str(DATA_YAML),
    "epochs":  STAGE2_EPOCHS,
    "imgsz":   IMG_SIZE,
    "batch":   -1,        # auto-batch
    "device":  DEVICE,
    "workers": WORKERS,

    # ── Kunci Stage 2: buka semua layer, LR kecil ────────────────────────────
    "freeze":        0,       # ← SEMUA layer terbuka
    "lr0":           0.0001,  # ← 10× lebih kecil dari Stage 1
    "lrf":           0.01,    # final LR = lr0 × lrf = 1e-6
    "momentum":      0.937,
    "weight_decay":  0.0005,
    "cos_lr":        True,
    "warmup_epochs": 1.0,     # singkat — weights sudah matang
    "warmup_momentum": 0.8,
    "warmup_bias_lr":  0.05,  # lebih kecil dari Stage 1

    # ── Optimizer ─────────────────────────────────────────────────────────────
    "optimizer": "AdamW",

    # ── Regularization & control ──────────────────────────────────────────────
    "patience":      25,
    "close_mosaic":  15,
    "amp":           True,
    "deterministic": True,
    "seed":          SEED,
    "save":          True,
    "save_period":   5,
    "plots":         True,
    "val":           True,

    # ── Segmentation-specific ─────────────────────────────────────────────────
    "overlap_mask": True,
    "mask_ratio":   4,

    # ── Augmentation (lebih kuat dari Stage 1 karena full fine-tuning) ────────
    "hsv_h":         0.015,
    "hsv_s":         0.70,
    "hsv_v":         0.40,
    "degrees":       8.0,    # sedikit lebih bebas dari Stage 1
    "translate":     0.12,
    "scale":         0.55,
    "shear":         3.0,
    "perspective":   0.0005,
    "flipud":        0.0,
    "fliplr":        0.5,
    "mosaic":        0.75,   # dikurangi dari 1.0 — model sudah kenal variasi
    "mixup":         0.15,   # dinaikkan — regularisasi lebih kuat
    "copy_paste":    0.40,   # dinaikkan — kelas minoritas
    "copy_paste_mode": "flip",
}

print("Stage 2 Config:")
print(json.dumps(STAGE2_CONFIG, indent=2, default=str))


## 5. Run Stage 2 baseline training

In [ ]:
S2_EXP_NAME = "yolo11l_seg_stage2_baseline"
S2_EXP_DIR  = RUNS_ROOT / S2_EXP_NAME
S2_LAST_PT  = S2_EXP_DIR / "weights" / "last.pt"

if RESUME and S2_LAST_PT.exists():
    print("Resuming Stage 2 from:", S2_LAST_PT)
    train_model = YOLO(str(S2_LAST_PT))
    resume_arg  = True
else:
    train_model = YOLO(str(STAGE1_BEST))
    resume_arg  = False

s2_results = train_model.train(
    **STAGE2_CONFIG,
    project  = str(RUNS_ROOT),
    name     = S2_EXP_NAME,
    exist_ok = True,
    resume   = resume_arg,
    verbose  = True,
)

S2_RUN_DIR   = Path(train_model.trainer.save_dir)
STAGE2_BEST  = S2_RUN_DIR / "weights" / "best.pt"
STAGE2_LAST  = S2_RUN_DIR / "weights" / "last.pt"

assert STAGE2_BEST.exists(), f"Stage 2 best.pt tidak ditemukan: {STAGE2_BEST}"
print("\n" + "="*72)
print("STAGE 2 BASELINE DONE")
print("="*72)
print("Best  :", STAGE2_BEST)
print("Last  :", STAGE2_LAST)


In [ ]:
# ── Plot training history Stage 2 ────────────────────────────────────────────
results_csv = S2_RUN_DIR / "results.csv"
if results_csv.exists():
    hist = pd.read_csv(results_csv)
    hist.columns = [c.strip() for c in hist.columns]
    display(hist.tail(10))

    loss_cols = [c for c in hist.columns if "loss" in c.lower()]
    map_cols  = [c for c in hist.columns if "map" in c.lower()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for col in loss_cols:
        axes[0].plot(hist["epoch"], hist[col], label=col)
    axes[0].set_title("Stage 2 Loss"); axes[0].legend(fontsize=7); axes[0].grid(True)
    for col in map_cols:
        axes[1].plot(hist["epoch"], hist[col], label=col)
    axes[1].set_title("Stage 2 mAP"); axes[1].legend(fontsize=7); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(S2_RUN_DIR / "stage2_baseline_curves.png", dpi=100)
    plt.show()


## 6. Optuna hyperparameter search

Search dilakukan dari `STAGE2_BEST.pt` (bukan dari Stage 1) karena:
- Weights sudah dalam kondisi fine-tuned → trial lebih stabil dan representatif
- Tiap trial hanya 20 epoch → cukup untuk melihat tren konvergensi
- Search space difokuskan pada: `lr0`, `lrf`, `mosaic`, `copy_paste`,
  `mixup`, `weight_decay`, `loss gains` (box/cls/seg)

Optuna akan mencari kombinasi yang memaksimalkan `mask mAP50-95` di val set.


In [ ]:
OPTUNA_RUNS = RUNS_ROOT / "optuna_trials"
OPTUNA_RUNS.mkdir(parents=True, exist_ok=True)

def optuna_objective(trial):
    """Satu trial Optuna: train dari STAGE2_BEST dengan hyperparams yang disugest."""

    # ── Search space ──────────────────────────────────────────────────────────
    lr0          = trial.suggest_float("lr0",          1e-5,  5e-4,  log=True)
    lrf          = trial.suggest_float("lrf",          0.005, 0.05)
    weight_decay = trial.suggest_float("weight_decay", 1e-5,  1e-3,  log=True)
    mosaic       = trial.suggest_float("mosaic",       0.5,   1.0)
    copy_paste   = trial.suggest_float("copy_paste",   0.2,   0.6)
    mixup        = trial.suggest_float("mixup",        0.0,   0.25)
    box_gain     = trial.suggest_float("box",          4.0,   10.0)
    cls_gain     = trial.suggest_float("cls",          0.3,   1.5)
    seg_gain     = trial.suggest_float("dfl",          1.0,   3.0)

    trial_name = f"trial_{trial.number:03d}"
    trial_dir  = OPTUNA_RUNS / trial_name

    trial_config = {
        "data":    str(DATA_YAML),
        "epochs":  OPTUNA_EPOCHS,
        "imgsz":   IMG_SIZE,
        "batch":   -1,
        "device":  DEVICE,
        "workers": WORKERS,

        "freeze":        0,
        "lr0":           lr0,
        "lrf":           lrf,
        "momentum":      0.937,
        "weight_decay":  weight_decay,
        "cos_lr":        True,
        "warmup_epochs": 1.0,
        "optimizer":     "AdamW",

        "patience":      OPTUNA_PATIENCE,
        "amp":           True,
        "deterministic": False,   # False agar trial lebih cepat
        "seed":          SEED + trial.number,
        "save":          False,   # jangan simpan checkpoint tiap trial — hemat disk
        "val":           True,
        "plots":         False,
        "verbose":       False,

        "overlap_mask":  True,
        "mask_ratio":    4,

        # Augmentation yang dicari
        "mosaic":        mosaic,
        "copy_paste":    copy_paste,
        "mixup":         mixup,
        "copy_paste_mode": "flip",
        "hsv_h": 0.015, "hsv_s": 0.70, "hsv_v": 0.40,
        "degrees": 8.0, "translate": 0.12, "scale": 0.55,
        "shear": 3.0, "perspective": 0.0005,
        "flipud": 0.0, "fliplr": 0.5,

        # Loss gains
        "box": box_gain,
        "cls": cls_gain,
        "dfl": seg_gain,
    }

    try:
        trial_model = YOLO(str(STAGE2_BEST))
        trial_results = trial_model.train(
            **trial_config,
            project  = str(OPTUNA_RUNS),
            name     = trial_name,
            exist_ok = True,
        )
        # Ambil mask mAP50-95 sebagai objective
        val_metrics = trial_model.trainer.validator.metrics
        mask_map    = float(getattr(val_metrics.seg, "map", 0.0))
        print(f"  Trial {trial.number:3d} | lr0={lr0:.2e} mosaic={mosaic:.2f} "
              f"copy_paste={copy_paste:.2f} mixup={mixup:.2f} → mask_mAP50-95={mask_map:.4f}")
        return mask_map
    except Exception as e:
        print(f"  Trial {trial.number:3d} FAILED: {e}")
        return 0.0

print(f"Memulai Optuna search: {N_TRIALS} trials × {OPTUNA_EPOCHS} epochs")
print(f"Starting weights: {STAGE2_BEST}")
print()

study = optuna.create_study(
    direction = "maximize",
    sampler   = optuna.samplers.TPESampler(seed=SEED),
    pruner    = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
    study_name= "nutrivision_stage2_optuna",
)
study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n" + "="*72)
print("OPTUNA SEARCH DONE")
print("="*72)
print(f"Best trial : #{study.best_trial.number}")
print(f"Best value : mask_mAP50-95 = {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Simpan hasil Optuna
optuna_results = pd.DataFrame([
    {"trial": t.number, "value": t.value, **t.params}
    for t in study.trials if t.value is not None
]).sort_values("value", ascending=False)
optuna_results.to_csv(RUNS_ROOT / "optuna_results.csv", index=False)
display(optuna_results.head(10))


In [ ]:
# ── Visualisasi Optuna ────────────────────────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Trial values
    trial_nums = [t.number for t in study.trials if t.value is not None]
    trial_vals = [t.value  for t in study.trials if t.value is not None]
    axes[0].plot(trial_nums, trial_vals, "o-", color="#2a78d6")
    axes[0].axhline(study.best_value, color="#1baf7a", linestyle="--",
                    label=f"best={study.best_value:.4f}")
    axes[0].set_title("Optuna trial values"); axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("mask mAP50-95"); axes[0].legend(); axes[0].grid(True)

    # Parameter importance (jika tersedia)
    try:
        importances = optuna.importance.get_param_importances(study)
        axes[1].barh(list(importances.keys()), list(importances.values()), color="#eb6834")
        axes[1].set_title("Parameter importance"); axes[1].set_xlabel("Importance score")
    except Exception:
        axes[1].text(0.5, 0.5, "Importance tidak tersedia\n(butuh ≥5 trials)",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(RUNS_ROOT / "optuna_visualization.png", dpi=100)
    plt.show()
except Exception as e:
    print(f"Visualisasi Optuna skip: {e}")


## 7. Final run dengan best Optuna params

Final run dimulai dari `STAGE2_BEST.pt` (bukan dari pretrained),
menggunakan hyperparams terbaik dari Optuna.
Ini adalah model yang akan digunakan untuk evaluasi dan deployment.


In [ ]:
best_p = study.best_params

FINAL_CONFIG = {
    # ── Dataset & hardware ────────────────────────────────────────────────────
    "data":    str(DATA_YAML),
    "epochs":  FINAL_EPOCHS,
    "imgsz":   IMG_SIZE,
    "batch":   -1,
    "device":  DEVICE,
    "workers": WORKERS,

    # ── Best params dari Optuna ───────────────────────────────────────────────
    "freeze":        0,
    "lr0":           best_p.get("lr0",          0.0001),
    "lrf":           best_p.get("lrf",          0.01),
    "momentum":      0.937,
    "weight_decay":  best_p.get("weight_decay", 0.0005),
    "cos_lr":        True,
    "warmup_epochs": 1.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr":  0.05,
    "optimizer":     "AdamW",

    # ── Augmentation terbaik ──────────────────────────────────────────────────
    "mosaic":          best_p.get("mosaic",      0.75),
    "copy_paste":      best_p.get("copy_paste",  0.40),
    "mixup":           best_p.get("mixup",       0.15),
    "copy_paste_mode": "flip",
    "hsv_h": 0.015, "hsv_s": 0.70, "hsv_v": 0.40,
    "degrees": 8.0, "translate": 0.12, "scale": 0.55,
    "shear": 3.0, "perspective": 0.0005,
    "flipud": 0.0, "fliplr": 0.5,

    # ── Loss gains dari Optuna ────────────────────────────────────────────────
    "box": best_p.get("box", 7.5),
    "cls": best_p.get("cls", 0.5),
    "dfl": best_p.get("dfl", 1.5),

    # ── Control ───────────────────────────────────────────────────────────────
    "patience":      30,     # lebih sabar di final run
    "close_mosaic":  15,
    "amp":           True,
    "deterministic": True,
    "seed":          SEED,
    "save":          True,
    "save_period":   5,
    "plots":         True,
    "val":           True,
    "overlap_mask":  True,
    "mask_ratio":    4,
}

print("Final Run Config:")
print(json.dumps(FINAL_CONFIG, indent=2, default=str))


In [ ]:
FINAL_EXP_NAME = "yolo11l_seg_stage2_final"

final_model    = YOLO(str(STAGE2_BEST))
final_results  = final_model.train(
    **FINAL_CONFIG,
    project  = str(RUNS_ROOT),
    name     = FINAL_EXP_NAME,
    exist_ok = True,
    verbose  = True,
)

FINAL_RUN_DIR  = Path(final_model.trainer.save_dir)
FINAL_BEST     = FINAL_RUN_DIR / "weights" / "best.pt"
FINAL_LAST     = FINAL_RUN_DIR / "weights" / "last.pt"

assert FINAL_BEST.exists(), f"Final best.pt tidak ditemukan: {FINAL_BEST}"
print("\n" + "="*72)
print("FINAL RUN DONE")
print("="*72)
print("Final best:", FINAL_BEST)


In [ ]:
# ── Plot final run history ────────────────────────────────────────────────────
final_csv = FINAL_RUN_DIR / "results.csv"
if final_csv.exists():
    hist_f = pd.read_csv(final_csv)
    hist_f.columns = [c.strip() for c in hist_f.columns]
    display(hist_f.tail(10))

    loss_cols = [c for c in hist_f.columns if "loss" in c.lower()]
    map_cols  = [c for c in hist_f.columns if "map"  in c.lower()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for col in loss_cols:
        axes[0].plot(hist_f["epoch"], hist_f[col], label=col)
    axes[0].set_title("Final Run Loss"); axes[0].legend(fontsize=7); axes[0].grid(True)
    for col in map_cols:
        axes[1].plot(hist_f["epoch"], hist_f[col], label=col)
    axes[1].set_title("Final Run mAP"); axes[1].legend(fontsize=7); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(FINAL_RUN_DIR / "final_run_curves.png", dpi=100)
    plt.show()


## 8. Final evaluation — val dan test split

In [ ]:
EVAL_ROOT = WORK_ROOT / "stage2_evaluation"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

# Eval YAML selalu pakai folder biasa (bukan manifest oversampled)
EVAL_DATA_YAML = WORK_ROOT / "stage2_eval_data.yaml"
eval_cfg = {
    "path":  str(DATASET_ROOT),
    "train": str(TRAIN_IMAGES),
    "val":   str(VAL_IMAGES),
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
if TEST_IMAGES:
    eval_cfg["test"] = str(TEST_IMAGES)
EVAL_DATA_YAML.write_text(yaml.safe_dump(eval_cfg, sort_keys=False, allow_unicode=True))

final_best_model = YOLO(str(FINAL_BEST))

def flatten_results(results, split_name):
    row = {"split": split_name}
    for k, v in (getattr(results, "results_dict", {}) or {}).items():
        try:    row[k] = float(v)
        except: row[k] = str(v)
    return row

eval_rows    = []
eval_results = {}

for split_name in ("val", "test"):
    if split_name == "test" and not TEST_IMAGES:
        continue
    print(f"\nEvaluating {split_name} ...")
    res = final_best_model.val(
        data     = str(EVAL_DATA_YAML),
        split    = split_name,
        imgsz    = IMG_SIZE,
        batch    = 4,
        device   = DEVICE,
        workers  = WORKERS,
        plots    = True,
        project  = str(EVAL_ROOT),
        name     = split_name,
        exist_ok = True,
        verbose  = True,
    )
    eval_results[split_name] = res
    eval_rows.append(flatten_results(res, split_name))

eval_df = pd.DataFrame(eval_rows)
display(eval_df)
eval_df.to_csv(EVAL_ROOT / "stage2_split_metrics.csv", index=False)


In [ ]:
# ── Per-class metrics ─────────────────────────────────────────────────────────
def per_class_seg_metrics(results, class_names):
    seg = getattr(results, "seg", None)
    if seg is None: return pd.DataFrame()
    indices = getattr(seg, "ap_class_index", None)
    if indices is None: return pd.DataFrame()
    indices = [int(i) for i in np.asarray(indices).tolist()]
    p_arr = np.asarray(getattr(seg, "p",    []))
    r_arr = np.asarray(getattr(seg, "r",    []))
    ap50  = np.asarray(getattr(seg, "ap50", []))
    ap    = np.asarray(getattr(seg, "ap",   []))
    rows = []
    for pos, cls_id in enumerate(indices):
        row = {"class_id": cls_id,
               "class_name": class_names[cls_id] if cls_id < len(class_names) else str(cls_id)}
        for name, arr in [("precision",p_arr),("recall",r_arr),("mAP50",ap50),("mAP50-95",ap)]:
            if pos < len(arr): row[name] = float(arr[pos])
        if "precision" in row and "recall" in row:
            d = row["precision"] + row["recall"]
            row["f1"] = 2*row["precision"]*row["recall"]/d if d > 0 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

all_pc = []
for split_name, res in eval_results.items():
    tbl = per_class_seg_metrics(res, CLASS_NAMES)
    if not tbl.empty:
        tbl.insert(0, "split", split_name)
        all_pc.append(tbl)

if all_pc:
    pc_df = pd.concat(all_pc, ignore_index=True).round(4)
    display(pc_df.sort_values(["split", "mAP50-95"]))
    pc_df.to_csv(EVAL_ROOT / "stage2_per_class_metrics.csv", index=False)

    # Recall per class — metrik utama NutriVision
    test_pc = pc_df[pc_df["split"]=="test"].sort_values("recall")
    if not test_pc.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = ["#e34948" if v < 0.5 else "#f5a623" if v < 0.7 else "#1baf7a"
                  for v in test_pc["recall"]]
        ax.barh(test_pc["class_name"], test_pc["recall"], color=colors)
        ax.axvline(0.5, color="red",    linestyle="--", linewidth=1, label="0.50")
        ax.axvline(0.7, color="orange", linestyle="--", linewidth=1, label="0.70")
        ax.set_xlabel("Recall")
        ax.set_title("Stage 2 Final — Per-class Recall (test set)")
        ax.legend(); plt.tight_layout()
        plt.savefig(EVAL_ROOT / "stage2_recall_per_class.png", dpi=100)
        plt.show()


## 9. Save artifacts dan download

In [ ]:
# ── Prediction preview ────────────────────────────────────────────────────────
if TEST_IMAGES:
    test_imgs = sorted(p for p in TEST_IMAGES.rglob("*")
                       if p.suffix.lower() in IMAGE_EXTS)
    previews = test_imgs[:min(24, len(test_imgs))]
    if previews:
        final_best_model.predict(
            source   = [str(p) for p in previews],
            conf     = 0.25,
            imgsz    = IMG_SIZE,
            device   = DEVICE,
            save     = True,
            project  = str(EVAL_ROOT),
            name     = "test_predictions",
            exist_ok = True,
            verbose  = False,
        )

# ── Summary JSON ──────────────────────────────────────────────────────────────
summary = {
    "pipeline": {
        "stage1_best":      str(STAGE1_BEST),
        "stage2_best":      str(STAGE2_BEST),
        "final_best":       str(FINAL_BEST),
    },
    "optuna": {
        "n_trials":         N_TRIALS,
        "best_trial":       study.best_trial.number,
        "best_mask_map":    round(study.best_value, 4),
        "best_params":      study.best_params,
    },
    "final_config":         {k: str(v) if isinstance(v, Path) else v
                             for k, v in FINAL_CONFIG.items()},
    "evaluation":           eval_df.to_dict(orient="records"),
    "gpu":                  GPU_NAME,
    "gpu_memory_gb":        round(GPU_MEMORY_GB, 2),
}
SUMMARY_JSON = EVAL_ROOT / "stage2_summary.json"
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

# ── ZIP all artifacts ─────────────────────────────────────────────────────────
ARTIFACT_ZIP = WORK_ROOT / "stage2_artifacts.zip"
if ARTIFACT_ZIP.exists():
    ARTIFACT_ZIP.unlink()

with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    # Final best model
    archive.write(FINAL_BEST,  "weights/final_best.pt")
    archive.write(STAGE2_BEST, "weights/stage2_baseline_best.pt")
    # Evaluation
    for p in EVAL_ROOT.rglob("*"):
        if p.is_file() and p.suffix in {".csv", ".json", ".png"}:
            archive.write(p, Path("evaluation") / p.relative_to(EVAL_ROOT))
    # Optuna
    optuna_csv = RUNS_ROOT / "optuna_results.csv"
    if optuna_csv.exists():
        archive.write(optuna_csv, "optuna/optuna_results.csv")
    optuna_png = RUNS_ROOT / "optuna_visualization.png"
    if optuna_png.exists():
        archive.write(optuna_png, "optuna/optuna_visualization.png")

print("="*72)
print("STAGE 2 + OPTUNA COMPLETE")
print("="*72)
print("Final best model  :", FINAL_BEST)
print("Stage 2 baseline  :", STAGE2_BEST)
print("Artifact ZIP      :", ARTIFACT_ZIP)
print("\nLangkah selanjutnya:")
print("  1. Commit notebook ini (Save & Run All)")
print("  2. Download stage2_artifacts.zip dari Output tab")
print("  3. Jalankan 07_final_evaluation.ipynb dengan final_best.pt")


In [ ]:
display(FileLink(str(ARTIFACT_ZIP)))
display(FileLink(str(FINAL_BEST)))
display(FileLink(str(EVAL_ROOT / "stage2_split_metrics.csv")))
display(FileLink(str(EVAL_ROOT / "stage2_per_class_metrics.csv")))
display(FileLink(str(SUMMARY_JSON)))


---
## Checklist setelah Stage 2

- [ ] `final_best.pt` muncul di Output tab
- [ ] `stage2_artifacts.zip` berhasil didownload
- [ ] Mask recall test set > 0.40 (target minimum)
- [ ] Mean class coverage > 85% di test set
- [ ] Tidak ada kelas dengan recall = 0.00
- [ ] Optuna best value lebih tinggi dari Stage 2 baseline

Jika target belum tercapai → cek `optuna_results.csv` untuk pattern params,
pertimbangkan tambah data kelas lemah (`tofu`, `sambal`, `squid`), atau naikkan
`N_TRIALS` dan jalankan Optuna lebih lama.
